# CLIP text and image embeddings

This notebook encodes 32 validation image-caption pairs with frozen CLIP ViT-B/32. Its text and image encoders produce projected, normalized vectors in a shared embedding space.


In [1]:
import json
from IPython.display import display
import p2_pipeline as p2
# Record package versions and GPU availability.
environment = p2.check_environment(save_requirements=True)
print(json.dumps(environment, indent=2))
assert environment["ready"], "Environment validation failed; details are recorded above."
config = p2.load_config("config.json")


{
  "utc": "2026-10-25T08:43:05.773483+00:00",
  "python": "3.10.16",
  "python_executable": "/opt/miniconda/envs/eai/bin/python",
  "platform": "Linux-5.14.0-362.13.1.el9_3.x86_64-x86_64-with-glibc2.35",
  "packages": {
    "torch": "2.5.1",
    "diffusers": "0.32.2",
    "transformers": "4.49.0",
    "accelerate": "1.5.2",
    "huggingface-hub": "0.29.3",
    "safetensors": "0.5.3",
    "numpy": "1.26.4",
    "Pillow": "11.1.0",
    "ipykernel": "6.29.5",
    "ipython": "8.34.0",
    "nbconvert": "7.16.6"
  },
  "git_commit": "453f6f78b079ab536a95375c2d4f8b5027f41d46",
  "source_sha256": "fc927d13d1006effe88166bed6bd821f66b662cc3c3e15150590182e8e0fff0b",
  "torch_cuda": "12.1",
  "cudnn": 90100,
  "cuda_available": true,
  "gpu": "Tesla V100-SXM2-32GB",
  "gpu_total_gib": 31.7393798828125,
  "missing_packages": [],
  "import_errors": {},
  "ready": true,
  "requirements_saved": true
}


In [2]:
import numpy as np
import torch
print("Device:", p2.device_for(config))
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", p2.verify_dataset(config)["splits"])


Device: cuda
GPU: Tesla V100-SXM2-32GB
Dataset: {'train': 1000, 'validation': 100, 'test': 100}


## Encode paired captions and images

The model and processor use the tokenizer revision recorded during data preparation. Caption lengths are checked before encoding; feature vectors are checked for finite values and unit length.


In [3]:
embedding_summary = p2.embed(config)
display(embedding_summary)
# Paired modalities must share the same projected feature dimensions.
assert embedding_summary["sample_count"] == 32
assert embedding_summary["image_shape"] == embedding_summary["text_shape"]


2026-10-25 04:43:12.369958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1792917792.390353 3003987 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1792917792.396553 3003987 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792917792.413196 3003987 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792917792.413214 3003987 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792917792.413217 3003987 computation_placer.cc:177] computation placer alr

{'status': 'complete',
 'sample_count': 32,
 'text_shape': [32, 512],
 'image_shape': [32, 512],
 'paired_cosine_mean': 0.29797011613845825,
 'off_diagonal_cosine_mean': 0.1459401696920395,
 'sample_text_to_image_top1': 0.9375,
 'seconds': 11.90028490871191,
 'device': 'cuda',
 'gpu': 'Tesla V100-SXM2-32GB',
 'scope': 'sanity check on selected validation pairs; not held-out generation evaluation',
 'artifact_hashes': {'embeddings.npz': '43dc5cb46476e0620bdcb31a33f4df21a74a390d3103d124a2b111bc46c67b16',
  'embedding_tokens.json': '69aa70682d1c841f5e9ce2ddd455906193e6c631b60dfc65d0236218fcee89eb'},
 'runtime': {'utc': '2026-10-25T08:43:33.076172+00:00',
  'python': '3.10.16',
  'python_executable': '/opt/miniconda/envs/eai/bin/python',
  'platform': 'Linux-5.14.0-362.13.1.el9_3.x86_64-x86_64-with-glibc2.35',
  'packages': {'torch': '2.5.1',
   'diffusers': '0.32.2',
   'transformers': '4.49.0',
   'accelerate': '1.5.2',
   'huggingface-hub': '0.29.3',
   'safetensors': '0.5.3',
   'numpy

## Inspect tokenization and similarity

Token counts exclude padding, while saved `input_ids` retain the padded batch sequences. Attention masks identify padding positions. Cosine similarity is the dot product of unit feature vectors.

Paired/unpaired similarities and top-1 retrieval measure reference-pair alignment within these 32 examples. Similar captions or images can cause mismatches; these statistics do not measure generated-image quality.


In [4]:
run = p2.local_path(config, "run_dir")
tokens = p2.read_json(run / "embedding_tokens.json")
display(tokens[:3])
with np.load(run / "embeddings.npz") as saved:
    cosine = saved["cosine_similarity"]
    image_ids = saved["image_ids"]
    print("Similarity shape:", cosine.shape)
    print("First 5×5 cosine matrix:\n", np.round(cosine[:5, :5], 3))
    for i in range(min(5, len(cosine))):
        winner = int(cosine[i].argmax())
        print("caption image", int(image_ids[i]), "retrieved image", int(image_ids[winner]),
              "paired cosine", round(float(cosine[i, i]), 3))


[{'image_id': 331352,
  'caption': 'A small closed toilet in a cramped space.',
  'input_ids': [49406,
   320,
   2442,
   4552,
   11071,
   530,
   320,
   1075,
   22640,
   2138,
   269,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407],
  'token_count': 12},
 {'image_id': 519491,
  'caption': 'A tall clock tower with a statue on top.',
  'input_ids': [49406,
   320,
   7771,
   6716,
   4730,
   593,
   320,
   8222,
   525,
   1253,
   269,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407],
  'token_count': 12},
 {'image_id': 61584,
  'caption': 'a couple is sitting on a statue of a horse and some plants',
  'input_ids': [49406,
   320,
   3377,
   533,
   4919,
   525,
   320,
   8222,
   539,
   320,
   4558,
   537,
   836,
   5829,
   49407,
   49407,
   49407,
   49407,
   49407,
   49407],
  'token_count': 15}]

Similarity shape: (32, 32)
First 5×5 cosine matrix:
 [[0.293 0.136 0.205 0.161 0.167]
 [0.103 0.291 0.155 0.155 0.142]
 [0.138 0.177 0.289 0.157 0.183]
 [0.165 0.123 0.154 0.244 0.156]
 [0.105 0.159 0.181 0.121 0.302]]
caption image 331352 retrieved image 331352 paired cosine 0.293
caption image 519491 retrieved image 519491 paired cosine 0.291
caption image 61584 retrieved image 61584 paired cosine 0.289
caption image 486040 retrieved image 486040 paired cosine 0.244
caption image 181542 retrieved image 181542 paired cosine 0.302


## Text conditioning in Stable Diffusion

Stable Diffusion 1.5 uses its own tokenizer and Transformer text encoder to supply token-level states to U-Net cross-attention. The pooled CLIP ViT-B/32 vectors above are used only for the embedding check.

Features and diagnostics are saved in `embeddings.npz`, `embedding_tokens.json` and `embedding_summary.json`.
